# Titanic Survival Prediction

Predict Titanic survival using feature engineering, missing-value handling, categorical encoding, model comparison, and feature importance.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import joblib
from preprocessing import TitanicFeatureEngineer


In [ ]:
df = pd.read_csv('data/titanic.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print(df.isnull().sum())
sns.countplot(data=df, x='Sex', hue='Survived')
plt.title('Survival by Sex')
plt.show()

In [ ]:
X = df.drop(columns='Survived')
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

numeric = ['Pclass','Age','SibSp','Parch','Fare','FamilySize','IsAlone','CabinKnown']
categorical = ['Sex','Embarked','Title']
preprocess = ColumnTransformer([
 ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric),
 ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical)
])
models = {
 'Logistic Regression': LogisticRegression(max_iter=1000),
 'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
 'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}
results = {}
for name, model in models.items():
 pipe = Pipeline([('features', TitanicFeatureEngineer()), ('preprocess', preprocess), ('model', model)])
 pipe.fit(X_train, y_train)
 pred = pipe.predict(X_test)
 results[name] = [accuracy_score(y_test,pred), precision_score(y_test,pred), recall_score(y_test,pred), f1_score(y_test,pred)]
 print(name, results[name])


In [ ]:
results_df = pd.DataFrame(results, index=['Accuracy','Precision','Recall','F1']).T
results_df

In [ ]:
best_pipe = Pipeline([('features', TitanicFeatureEngineer()), ('preprocess', preprocess), ('model', RandomForestClassifier(n_estimators=200, random_state=42))])
best_pipe.fit(X_train, y_train)
pred = best_pipe.predict(X_test)
print(classification_report(y_test, pred))
sns.heatmap(confusion_matrix(y_test,pred), annot=True, fmt='d')
plt.title('Random Forest Confusion Matrix')
plt.show()

In [ ]:
feature_names = best_pipe.named_steps['preprocess'].get_feature_names_out()
importance = best_pipe.named_steps['model'].feature_importances_
fi = pd.Series(importance, index=feature_names).sort_values(ascending=False).head(15)
fi.sort_values().plot(kind='barh')
plt.title('Top Random Forest Feature Importances')
plt.show()
fi

In [ ]:
joblib.dump(best_pipe, 'models/titanic_best_model.joblib', compress=3)
example = X_test.iloc[[0]]
print('Predicted survival:', int(best_pipe.predict(example)[0]))
print('Survival probability:', float(best_pipe.predict_proba(example)[0,1]))

Model saved to models/titanic_best_model.joblib